# LLM AUTONOMOUS AGENT - Website Finder v2.5
Production-ready architecture s:
- Agent state management
- Dynamic prompt engineering
- LLM-as-Judge verification
- Comprehensive metrics
- Memory for topics in json file


In [1]:
import os
import json
import requests
import pandas as pd
import asyncio
import getpass
from bs4 import BeautifulSoup
import re
from urllib.parse import urlparse
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple
from enum import Enum
from recommendation_research_llm_handler.llm_client import LLMHandler

In [2]:
import nest_asyncio
nest_asyncio.apply()

# 1. DATA MODELS

In [3]:
class VerdictType(Enum):
    APPROVE = "APPROVE"
    REJECT = "REJECT"
    UNCLEAR = "UNCLEAR"


@dataclass
class URLAttempt:
    """Záznam jednoho pokusu na URL"""
    topic: str
    url: str
    success: bool
    status_code: Optional[int]
    error_message: Optional[str] = None
    content_length: int = 0
    ad_score: int = 0


@dataclass
class AuditDecision:
    """Rozhodnutí auditora (LLM)"""
    url: str
    verdict: VerdictType
    verdict_confidence: float
    primary_reason: str
    website_type: str  # Blog, Forum, Eshop, Generalist, Inactive
    specialization_score: float  # 0-1
    content_age_months: Optional[int] = None
    reasoning_detail: str = ""


@dataclass
class JudgeReview:
    """Recenze Judge modulu na auditní rozhodnutí"""
    audit_verdict: VerdictType
    judge_verdict: VerdictType
    judge_confidence: float
    agrees_with_auditor: bool
    judge_reasoning: str
    recommended_action: str  # APPROVE_FINAL, REJECT_FINAL, NEEDS_MANUAL_REVIEW


@dataclass
class AgentState:
    """Stav agenta během běhu"""
    current_topic: str = ""
    urls_processed: int = 0
    urls_approved: int = 0
    current_batch_index: int = 0
    adaptation_history: List[str] = field(default_factory=list)
    common_rejection_reason: str = ""
    processed_domains: set = field(default_factory=set)


@dataclass
class MetricsSnapshot:
    """Snapshot metrik v čase"""
    timestamp: str
    topic_coverage_rate: float
    url_success_rate: float
    domain_diversity: float
    audit_judge_agreement: float
    false_positive_rate: float
    avg_audit_confidence: float
    api_calls_total: int
    estimated_cost: float

# 2. PROMPT ENGINEERING - STRUKTUROVANÉ A KONTEXTOVÉ

In [4]:
class PromptFactory:
    """Generuje dynamické prompty na základě kontextu"""
    
    @staticmethod
    def generate_niche_topics_prompt(count: int) -> str:
        """Vylepšený prompt s few-shot příklady"""
        return f"""
# ROLE
Jsi Senior Content Strategist a expert na identifikaci vysokoengagement hobby niš pro český trh.

# PŘÍKLADY SPRÁVNÝCH TÉMAT (Few-shot)

✅ SPRÁVNĚ - Specifická, s komunitou:
- "Fermentace zeleniny" (ne: "vaření")
- "Ultralight treking" (ne: "cestování")
- "FPV drony racing" (ne: "drony obecně")
- "Akvaristika s krevetami" (ne: "akvaristika")

❌ ŠPATNĚ - Příliš obecné:
- "Vaření" → Příliš broad
- "Sport" → Není niche
- "Cestování" → Bez specificity

# ÚKOL
Vygeneruj seznam {count} specifických hobby témat v češtině.

# KRITÉRIA VÝBĚRU
1. **Odbornost:** Vyžadují učení a skill (ne pasivní zábava)
2. **Vybavení:** Existuje trh se specifickými nástroji/hardwarem
3. **Komunita:** Lze prokázat online komunitu (fóra, YouTube, skupiny)
4. **Search Volume:** Téma je reálně hledané

# ANTI-HALLUCINATION CHECK
- Každé téma musí existovat jako python balíček nebo aspoň Wikipedia článek
- Pokud si není jisté, vynech to

# VÝSTUP
Vrať POUZE JSON bez komentářů:
[
    {{"topic": "Simracing", "rarity": 0.8, "expected_blogs": 20}},
    {{"topic": "Fermentace zeleniny", "rarity": 0.9, "expected_blogs": 5}}
]
"""

    @staticmethod
    def generate_seed_urls_prompt(topic: str, count: int, agent_state: AgentState) -> str:
        """Kontextový prompt se znalostem agenta"""
        context_info = ""
        if agent_state.common_rejection_reason:
            context_info = f"\n# AGENT INSIGHT\nPoslední weby na toto téma byly často zamítány jako: {agent_state.common_rejection_reason}\nTentokrát vyhledej weby, které to NEBUDOU mít.\n"
        
        return f"""
# ROLE & TASK
Jsi expert na vyhledávání kvalitních niche blogů. Tvůj úkol je najít {count} NEJLEPŠÍCH zdrojů pro téma: "{topic}"

# DOMAIN KNOWLEDGE - Co je kvalitní niche blog?
✓ Vlastní obsah (ne agregátor článků)
✓ Pravidelné updates (ne mrtvý web)
✓ Community engagement (komentáře, sociální média)
✓ Odborné hloubka (ne povrchní články)

# STRATEGIE HLEDÁNÍ
Pro "{topic}" hledej tyto typy webů:
- Špecifické technické blogy
- YouTubery s vlastními weby
- Komunitní portály (slušně spravované)
- Nezávislé autory (ne mediální dům)

# NEGATIVE EXAMPLES
IGNORUJ:
- Generální portály (Dama, iDnes, Novinky)
- E-shopy (primárně prodej)
- Fóra bez vlastního obsahu
- Weby starší 6 měsíců bez updates

{context_info}

# PŘÍKLADY SPRÁVNÉ ODPOVĚDI
Téma: "Fermentace zeleniny"
[
    {{"url": "https://www.wildfermentation.com/", "source_type": "blog", "confidence": 0.95}},
    {{"url": "https://fermentationculture.eu/", "source_type": "blog", "confidence": 0.9}}
]

# VÝSTUP - JSON s alespoň {count} URL
[
    {{"url": "https://...", "source_type": "blog|forum|wiki|youtube_articles", "confidence": 0.85}}
]
"""

    @staticmethod
    def audit_url_content_prompt(
        url: str, 
        topic: str, 
        content: str, 
        ad_score: int,
        agent_state: AgentState
    ) -> str:
        """Strukturovaný audit prompt s decision tree"""
        
        context_str = f"[{agent_state.urls_processed}/{agent_state.current_batch_index}] Dosavadní schváleno: {agent_state.urls_approved}"
        
        return f"""
# ROLE
Jsi striktní auditor obsahu s úkolem recenzovat web: {url} pro téma: "{topic}"

# AGENT CONTEXT
{context_str}

# OBSAH WEBU (prvních 3000 znaků)
{content[:3000]}

# AD SCORE
{ad_score}/10 (0=čistý, 10=plný reklam)

# ROZHODOVACÍ LOGIKA - FOLLOW THIS EXACTLY

## STEP 1: Je to obecný portál?
Hledej signály: "celebrity", "horoskop", "zprávy", "novinky", "magazín o všem"
Pokud NE → CONTINUE na STEP 2
Pokud ANO → REJECT (Reason: "Generalist portal, ne niche")

## STEP 2: Je to e-shop primárně?
Hledej: "cart", "checkout", "price", "objednat", "koupit"
Pokud >30% obsahu je prodej → REJECT (Reason: "Eshop/Corporate")
Pokud <30% → CONTINUE na STEP 3

## STEP 3: Je 100% zaměřený na "{topic}"?
Hledej články mimo "{topic}"
- Pokud <5% obsahu je mimo → CONTINUE
- Pokud >20% mimo → REJECT (Reason: "Недостаточная специализация")

## STEP 4: Je web aktivní (poslední 6 měsíců)?
Hledej data v article metadata nebo copyright
- Pokud starší → REJECT (Reason: "Inactive - no updates")
- Pokud aktuální → APPROVE

## STEP 5: Specialization Score
Ohodnoť 0-1 jak moc je web specializovaný:
- 0.95-1.0: Absolutně niche (např. jen Simracing)
- 0.85-0.94: Těžko zaměřený (např. Outdoor obecně, ale 80% ultralight)
- 0.70-0.84: Aspoň na správné téma
- <0.70: REJECT

# VÝSTUP - PARSEABLE JSON
{{
    "verdict": "APPROVE" | "REJECT",
    "verdict_confidence": 0.92,
    "website_type": "Blog" | "Forum" | "Eshop" | "Generalist" | "Inactive" | "Corporate",
    "primary_reason": "Vysvětlení v jedné větě",
    "specialization_score": 0.87,
    "content_age_months": 3,
    "for_judge_review": false,
    "decision_tree_path": "STEP 1→ NO, STEP 2→ NO, STEP 3→ YES, STEP 4→ YES, STEP 5→ 0.87"
}}
"""

    @staticmethod
    def judge_review_prompt(audit: AuditDecision, content: str) -> str:
        """Judge recenzuje auditní rozhodnutí"""
        return f"""
# ROLE
Jsi SENIOR AUDITOR - tvůj úkol je kontrolovat rozhodnutí auditora.
Mas přístup ke stejnému webu a musíš nezávisle soudit.

# ORIGINÁLNÍ AUDITNÍ ROZHODNUTÍ
Verdict: {audit.verdict.value}
Confidence: {audit.verdict_confidence}
Důvod: {audit.primary_reason}
Website Type: {audit.website_type}
Specialization Score: {audit.specialization_score}

# OBSAH WEBU (pro tvé posouzení)
{content[:2000]}

# TVŮJ ÚKOL
1. Souhlasíš s auditorem?
2. Pokud NE - jaké je SPRÁVNÉ rozhodnutí?
3. Jaké je tvé confidence? (0-1)

# DŮVOD - Pokud se lišíš:
Auditor řekl: {audit.verdict.value}
Ty říkáš: [APPROVE | REJECT | UNCLEAR]
Důvod: [tvůj případ]

# VÝSTUP JSON
{{
    "judge_verdict": "APPROVE" | "REJECT" | "UNCLEAR",
    "judge_confidence": 0.89,
    "agrees_with_auditor": true | false,
    "judge_reasoning": "Souhlasím protože...",
    "recommended_action": "APPROVE_FINAL" | "REJECT_FINAL" | "NEEDS_MANUAL_REVIEW"
}}
"""

# 3. CONTENT EXTRACTION - EXISTING

In [5]:
def extract_content_from_response(res) -> str:
    if isinstance(res, str):
        return res
    if isinstance(res, dict):
        return res.get('text') or str(res)
    if hasattr(res, 'choices'):
        try:
            return res.choices[0].message.content
        except:
            pass
    if hasattr(res, 'content'):
        return res.content
    return str(res)

In [6]:
def clean_json_response(text: str) -> str:
    text = re.sub(r'```json\n?', '', text)
    text = re.sub(r'```', '', text)
    return text.strip()


def extract_json_from_response(text: str) -> Dict or List:
    """Lépe extrahuj JSON"""
    if not text:
        return {}
    try:
        # Zkus najít JSON pole
        match = re.search(r'\[.*\]', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        # Zkus najít JSON objekt
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except:
        pass
    return {}

In [7]:
def analyze_ad_density(html_content: str) -> int:
    """Analýza reklam v obsahu"""
    ad_keywords = [
        "adsbygoogle", "sklik", "etarget", "adform", "doubleclick",
        "google_ads", "banner", "smartadserver", "criteo", "appnexus"
    ]
    raw_count = sum(html_content.count(kw) for kw in ad_keywords)
    return min(10, 1 + (raw_count // 2))

In [8]:
def get_page_content_smart(url: str) -> Tuple[bool, str, str, int]:
    """Aktualizovaná verze s lepším error handlingem"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    try:
        response = requests.get(url, headers=headers, timeout=8)
        
        # Fallback na root při 404
        if response.status_code == 404:
            root_url = f"{urlparse(url).scheme}://{urlparse(url).netloc}"
            if root_url != url.rstrip('/'):
                response = requests.get(root_url, headers=headers, timeout=8)
                url = root_url
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Minimální bortání
            for tag in soup(['script', 'style', 'svg', 'iframe', 'noscript', 'form']):
                tag.extract()
            
            text = soup.get_text(separator=' ').strip()
            text = re.sub(r'\s+', ' ', text)
            
            ad_score = analyze_ad_density(response.text)
            return True, url, text[:4000], ad_score
        
        return False, url, f"Status: {response.status_code}", 0
    
    except Exception as e:
        return False, url, str(e), 0

In [67]:
def canonicalize_blog_url(url: str) -> str:
    """
    Zkusí z článkové URL udělat homepage blogu / sekce.
    1) Podívá se do BLOG_ROOT_OVERRIDES
    2) Obecné vzory: /blog/, /blogs/news, /clanky, /articles, jazykové prefixy
    3) Fallback: domain root (/) 
    """
    parsed = urlparse(url)
    domain = parsed.netloc.lower()
    path = parsed.path or "/"

    # 1) Doménové overrides
    if domain in BLOG_ROOT_OVERRIDES:
        new_path = BLOG_ROOT_OVERRIDES[domain]
        return urlunparse(parsed._replace(path=new_path, query="", fragment=""))

    # 2) Generické vzory – necháme jen kořenovou sekci
    segments = [s for s in path.split("/") if s]

    # a) /blogs/news/... -> /blogs/news
    if len(segments) >= 2 and segments[0] in ("blog", "blogs"):
        new_path = "/" + "/".join(segments[:2])  # /blogs/news
        return urlunparse(parsed._replace(path=new_path, query="", fragment=""))

    # b) /blog/... nebo /clanky/... nebo /articles/... -> jen ta složka
    blog_like = {"blog", "clanky", "clánky", "articles", "news"}
    if segments and segments[0].lower() in blog_like:
        new_path = f"/{segments[0]}/"
        return urlunparse(parsed._replace(path=new_path, query="", fragment=""))

    # c) jazykové prefixy typu /cs/... /en/... – necháme jazykový root
    if segments and segments[0] in {"cs", "en", "de", "fr"}:
        new_path = f"/{segments[0]}/"
        return urlunparse(parsed._replace(path=new_path, query="", fragment=""))

    # 3) fallback – domain root
    return urlunparse(parsed._replace(path="/", query="", fragment=""))

In [70]:
from urllib.parse import urlparse, urlunparse
import re

BLOG_ROOT_OVERRIDES = {
    # Prusa – česká verze blogu
    "blog.prusa3d.com": "/cs/",
    # Photographing Space – celý web je blog/revamp
    "www.photographingspace.com": "/",
    "photographingspace.com": "/",
    # Stefan Liebermann – blog sekce
    "www.stefanliebermann.de": "/blogs/news",
    "stefanliebermann.de": "/blogs/news",
}

In [42]:
from typing import List

class SearchProvider:
    async def search(self, topic: str, num_results: int) -> List[str]:
        raise NotImplementedError

In [74]:
import json
import os
from pathlib import Path
from datetime import datetime

class TopicsMemory:
    """Jednoduchá paměť na témata – JSON file."""
    
    def __init__(self, filepath: str = "topics_memory.json"):
        self.filepath = Path(filepath)
        self.data = self._load()
    
    def _load(self) -> dict:
        """Načti paměť z JSON."""
        if self.filepath.exists():
            with open(self.filepath, "r", encoding="utf-8") as f:
                return json.load(f)
        return {"topics": {}, "last_updated": None}
    
    def _save(self):
        """Ulož paměť do JSON."""
        self.data["last_updated"] = datetime.now().isoformat()
        with open(self.filepath, "w", encoding="utf-8") as f:
            json.dump(self.data, f, indent=2, ensure_ascii=False)
    
    def add_topic(self, topic: str, urls: list[str], metadata: dict = None):
        """Přidej téma + URL do paměti."""
        if topic not in self.data["topics"]:
            self.data["topics"][topic] = {
                "urls": urls,
                "count": len(urls),
                "created_at": datetime.now().isoformat(),
                "metadata": metadata or {}
            }
            self._save()
            print(f"  💾 [{topic}] Saved to memory ({len(urls)} URLs)")
    
    def has_topic(self, topic: str) -> bool:
        """Zkontroluj, zda je téma v paměti."""
        return topic in self.data["topics"]
    
    def get_topics(self) -> list[str]:
        """Vrať seznam všech zapamatovaných témat."""
        return list(self.data["topics"].keys())
    
    def get_urls_for_topic(self, topic: str) -> list[str]:
        """Vrať URL pro konkrétní téma."""
        if topic in self.data["topics"]:
            return self.data["topics"][topic].get("urls", [])
        return []
    
    def clear(self):
        """Vymažte paměť."""
        self.data = {"topics": {}, "last_updated": None}
        self._save()
        print("  🗑️ Memory cleared")
    
    def show_stats(self):
        """Zobraz statistiku paměti."""
        print(f"\n📚 Memory Stats:")
        print(f"  Total topics: {len(self.data['topics'])}")
        total_urls = sum(len(t.get("urls", [])) for t in self.data["topics"].values())
        print(f"  Total URLs: {total_urls}")
        if self.data["topics"]:
            print(f"\n  Topics:")
            for topic, info in self.data["topics"].items():
                print(f"    - {topic}: {info['count']} URLs")

# Inicializace
memory = TopicsMemory("topics_memory.json")
print(f"✅ Memory loaded: {len(memory.get_topics())} topics")

✅ Memory loaded: 0 topics


In [48]:
os.environ['SELLMA_API_KEY'] = getpass.getpass("Api klíč pro LLM proxy:")
LLM_API_KEY = os.environ['SELLMA_API_KEY']

Api klíč pro LLM proxy: ········


In [47]:
SERPAPI_KEY = "55ad0f604d92b00c71297fd841434a1239fb583074d380c1d60bcb2d7809e48e"

In [46]:
from autonomous_agent_with_serpapi_FIXED import SerpApiSearcher

class SerpApiSearchProvider(SearchProvider):
    def __init__(self, api_key: str):
        self.client = SerpApiSearcher(api_key)

    async def search(self, topic: str, num_results: int = 30) -> List[str]:
        rows = await self.client.search_topic(topic=topic, num_results=num_results)
        return [r["url"] for r in rows]

# 4. AGENT KERNEL

In [97]:
class WebFinderAgent:
    """Autonomous agent pro hledání niche webů."""

    def __init__(
        self,
        api_key: str,
        model: str = "gpt-5",
        judge_model: str = "gpt-5",
        search_provider: SearchProvider | None = None,
    ):
        # LLM klient (SELLMA proxy)
        self.llm_client = LLMHandler(api_key=api_key, model=model)
        self.judge_client = LLMHandler(api_key=api_key, model=judge_model)

        # Stav agenta
        self.state = AgentState()
        self.results: List[AuditDecision] = []
        self.metrics = MetricsTracker()
        self.prompt_factory = PromptFactory()

        # Volitelné: externí search provider (např. SerpApi)
        self.search_provider = search_provider

    async def generate_niche_topics(self, count: int) -> List[str]:
        """Generuj témata s adaptací."""
        print(f"🧠 [Agent] Generating {count} niche topics...")
        prompt = self.prompt_factory.generate_niche_topics_prompt(count)

        try:
            res = await self.llm_client.single_response(
                prompt=prompt,
                output_metadata=True,
                temperature=1,
            )
            content = extract_content_from_response(res)
            topics = extract_json_from_response(content)

            if isinstance(topics, list) and len(topics) > 0:
                if isinstance(topics[0], dict):
                    return [t.get("topic", t) for t in topics]
                return topics
            return []
        except Exception as e:
            print(f"❌ Topic generation failed: {e}")
            self.state.adaptation_history.append(f"Topic gen failed: {str(e)}")
            return []

    async def generate_seed_urls(self, topic: str, count: int) -> List[str]:
        """Generuj URLs s adaptací na základě state (LLM varianta)."""
        print(f" ↳ [Agent] Hledám zdroje pro: '{topic}'...")
        prompt = self.prompt_factory.generate_seed_urls_prompt(
            topic=topic,
            count=count,
            agent_state=self.state,
        )

        try:
            res = await self.llm_client.single_response(
                prompt=prompt,
                output_metadata=True,
                temperature=1,
            )
            content = extract_content_from_response(res)
            urls_data = extract_json_from_response(content)

            urls: List[str] = []
            if isinstance(urls_data, list):
                for item in urls_data:
                    if isinstance(item, str):
                        urls.append(item)
                    elif isinstance(item, dict):
                        urls.append(item.get("url", item))
            return urls[:count]
        except Exception as e:
            print(f"❌ URL generation failed: {e}")
            self.state.adaptation_history.append(f"URL gen failed for {topic}")
            return []

    async def audit_url_content(
        self,
        url: str,
        topic: str,
        content: str,
        ad_score: int,
    ) -> AuditDecision:
        """Audituj URL obsah s kontextem."""
        prompt = self.prompt_factory.audit_url_content_prompt(
            url=url,
            topic=topic,
            content=content,
            ad_score=ad_score,
            agent_state=self.state,
        )

        try:
            res = await self.llm_client.single_response(
                prompt=prompt,
                output_metadata=True,
                temperature=1,
            )
            content_str = extract_content_from_response(res)
            data = extract_json_from_response(content_str)

            verdict = VerdictType(data.get("verdict", "REJECT"))

            audit = AuditDecision(
                url=url,
                verdict=verdict,
                verdict_confidence=data.get("verdict_confidence", 0.5),
                primary_reason=data.get("primary_reason", "No reason"),
                website_type=data.get("website_type", "Unknown"),
                specialization_score=data.get("specialization_score", 0.0),
                content_age_months=data.get("content_age_months"),
                reasoning_detail=data.get("decision_tree_path", ""),
            )

            if audit.verdict == VerdictType.APPROVE:
                self.state.urls_approved += 1
            else:
                if self.state.common_rejection_reason != audit.website_type:
                    self.state.common_rejection_reason = audit.website_type

            return audit
        except Exception as e:
            print(f"❌ Audit failed: {e}")
            return AuditDecision(
                url=url,
                verdict=VerdictType.REJECT,
                verdict_confidence=0.0,
                primary_reason=f"Audit error: {str(e)}",
                website_type="ERROR",
                specialization_score=0.0,
            )

    async def judge_review(
        self,
        audit: AuditDecision,
        content: str,
    ) -> JudgeReview | None:
        """Judge recenzuje auditní rozhodnutí (optional pro hraniční případy)."""
        if not (0.4 <= audit.verdict_confidence <= 0.6):
            return None

        prompt = self.prompt_factory.judge_review_prompt(audit, content)

        try:
            res = await self.judge_client.single_response(
                prompt=prompt,
                output_metadata=True,
                temperature=1,
            )
            content_str = extract_content_from_response(res)
            data = extract_json_from_response(content_str)

            review = JudgeReview(
                audit_verdict=audit.verdict,
                judge_verdict=VerdictType(data.get("judge_verdict", "UNCLEAR")),
                judge_confidence=data.get("judge_confidence", 0.5),
                agrees_with_auditor=data.get("agrees_with_auditor", True),
                judge_reasoning=data.get("judge_reasoning", ""),
                recommended_action=data.get(
                    "recommended_action",
                    "NEEDS_MANUAL_REVIEW",
                ),
            )

            self.metrics.record_judge_review(review)
            return review
        except Exception as e:
            print(f"❌ Judge review failed: {e}")
            return None

    async def process_topic(
        self,
        topic: str,
        urls_per_topic: int = 30,
    ) -> List[AuditDecision]:
        """Zpracuj jedno téma (celý pipeline pro jedno topic)."""

        self.state.current_topic = topic
        self.state.urls_processed = 0
        self.state.urls_approved = 0
        self.state.common_rejection_reason = ""

        print(f"\n{'=' * 60}")
        print(f"📌 TOPIC: {topic.upper()}")
        print(f"{'=' * 60}")

        # 1) Získání URL – SerpApi pokud je k dispozici, jinak LLM seed-URLs
        if self.search_provider is not None:
            print("  🔎 [Search] Fetching URLs via SerpApi...")
            seed_urls = await self.search_provider.search(topic, urls_per_topic)
        else:
            print("  🔎 [Search] Generating URLs via LLM prompt...")
            seed_urls = await self.generate_seed_urls(topic, urls_per_topic)

        unique_urls = list(set(seed_urls))
        topic_results: List[AuditDecision] = []

        # 2) Procesuj jednotlivé URL
        for idx, url in enumerate(unique_urls, 1):
            self.state.current_batch_index = idx
            self.state.urls_processed = idx

            if any(
                pattern in url
                for pattern in ["facebook.com", "instagram.com", "twitter.com"]
            ):
                continue

            domain = urlparse(url).netloc.replace("www.", "")
            if domain in self.state.processed_domains:
                continue

            print(f" 🔎 [{idx}/{len(unique_urls)}] {url}...", end=" ", flush=True)

            success, final_url, content, ad_score = get_page_content_smart(url)
            if not success:
                print("❌ Nedostupné")
                continue

            # canonicalizace URL na blog/homepage
            canonical_url = canonicalize_blog_url(final_url)

            self.state.processed_domains.add(domain)

            # Blacklist check
            if blacklist.is_blacklisted(canonical_url):
                print("🚫 BLACKLISTED")
                continue

            audit = await self.audit_url_content(canonical_url, topic, content, ad_score)

            judge = await self.judge_review(audit, content)

            if judge and judge.recommended_action == "APPROVE_FINAL":
                audit.verdict = VerdictType.APPROVE
            elif judge and judge.recommended_action == "REJECT_FINAL":
                audit.verdict = VerdictType.REJECT

            icon = "✅" if audit.verdict == VerdictType.APPROVE else "⛔"
            print(f"{icon} {audit.website_type}")

            self.metrics.record_url_attempt(topic, canonical_url, audit)
            topic_results.append(audit)
            self.results.append(audit)

        return topic_results

    async def run_full_pipeline(
        self,
        num_topics: int = 10,
        urls_per_topic: int = 30,
        memory: TopicsMemory = None,
    ) -> pd.DataFrame:
        """Spusť celý pipeline: topics → URLs → audit → DF."""
        print("\n🚀 [Agent] Starting full pipeline...\n")

        # 1) Generování témat
        topics = await self.generate_niche_topics(num_topics)
        if not topics:
            print("❌ No topics generated")
            return pd.DataFrame()

        # Filtruj témata která už jsou v paměti
        if memory:
            existing = memory.get_topics()
            topics = [t for t in topics if t not in existing]
            print(f"✅ Generated {len(topics)} new topics (skipped {len(existing)} from memory)")
        else:
            print(f"✅ Generated {len(topics)} topics")

        if not topics:
            print("⚠️  All topics already in memory")
            return pd.DataFrame()

        print(f"  Topics: {', '.join(topics[:5])}...\n")

        # 2) Zpracování témat
        all_results: List[AuditDecision] = []
        for topic in topics:
            results = await self.process_topic(topic, urls_per_topic)
            all_results.extend(results)

            # Ulož do paměti hned po zpracování tématu
            if memory and results:
                urls = [r.url for r in results if r.verdict == VerdictType.APPROVE]
                if urls:
                    memory.add_topic(topic, urls, {"count": len(results)})

        # 3) Postprocessing → DataFrame
        df = self._postprocess_results(all_results)

        # 4) Výpis statistik
        self._print_final_stats(df)

        return df

    def _postprocess_results(self, results: List[AuditDecision]) -> pd.DataFrame:
        """Zpracuj výsledky do DataFrame."""
        if not results:
            return pd.DataFrame()

        rows = []
        for r in results:
            if r.verdict != VerdictType.APPROVE:
                continue
            domain = urlparse(r.url).netloc.replace("www.", "")
            rows.append(
                {
                    "topic": getattr(r, "topic", None),
                    "url": r.url,
                    "domain": domain,
                    "website_type": r.website_type,
                    "specialization_score": r.specialization_score,
                    "verdict_confidence": r.verdict_confidence,
                }
            )

        return pd.DataFrame(rows)

    def _print_final_stats(self, df: pd.DataFrame) -> None:
        """Jednoduchý výpis statistik nad výsledky."""
        if df.empty:
            print("❌ No approved URLs")
            return

        print("\n📊 FINAL STATS")
        print(f"  Total approved URLs: {len(df)}")
        print(f"  Avg specialization score: {df['specialization_score'].mean():.2f}")
        print(f"  By website_type:\n{df['website_type'].value_counts()}")

# 5. METRICS TRACKER

In [94]:
class WebFinderAgent:
    """Autonomous agent pro hledání niche webů."""

    def __init__(
        self,
        api_key: str,
        model: str = "gpt-5",
        judge_model: str = "gpt-5",
        search_provider: SearchProvider | None = None,
    ):
        # LLM klient (SELLMA proxy)
        self.llm_client = LLMHandler(api_key=api_key, model=model)
        self.judge_client = LLMHandler(api_key=api_key, model=judge_model)

        # Stav agenta
        self.state = AgentState()
        self.results: List[AuditDecision] = []
        self.metrics = MetricsTracker()
        self.prompt_factory = PromptFactory()

        # Volitelné: externí search provider (např. SerpApi)
        self.search_provider = search_provider

    async def generate_niche_topics(self, count: int) -> List[str]:
        """Generuj témata s adaptací."""
        print(f"🧠 [Agent] Generating {count} niche topics...")
        prompt = self.prompt_factory.generate_niche_topics_prompt(count)

        try:
            res = await self.llm_client.single_response(
                prompt=prompt,
                output_metadata=True,
                temperature=1,
            )
            content = extract_content_from_response(res)
            topics = extract_json_from_response(content)

            if isinstance(topics, list) and len(topics) > 0:
                if isinstance(topics[0], dict):
                    return [t.get("topic", t) for t in topics]
                return topics
            return []
        except Exception as e:
            print(f"❌ Topic generation failed: {e}")
            self.state.adaptation_history.append(f"Topic gen failed: {str(e)}")
            return []

    async def generate_seed_urls(self, topic: str, count: int) -> List[str]:
        """Generuj URLs s adaptací na základě state (LLM varianta)."""
        print(f" ↳ [Agent] Hledám zdroje pro: '{topic}'...")
        prompt = self.prompt_factory.generate_seed_urls_prompt(
            topic=topic,
            count=count,
            agent_state=self.state,
        )

        try:
            res = await self.llm_client.single_response(
                prompt=prompt,
                output_metadata=True,
                temperature=1,
            )
            content = extract_content_from_response(res)
            urls_data = extract_json_from_response(content)

            urls: List[str] = []
            if isinstance(urls_data, list):
                for item in urls_data:
                    if isinstance(item, str):
                        urls.append(item)
                    elif isinstance(item, dict):
                        urls.append(item.get("url", item))
            return urls[:count]
        except Exception as e:
            print(f"❌ URL generation failed: {e}")
            self.state.adaptation_history.append(f"URL gen failed for {topic}")
            return []

    async def audit_url_content(
        self,
        url: str,
        topic: str,
        content: str,
        ad_score: int,
    ) -> AuditDecision:
        """Audituj URL obsah s kontextem."""
        prompt = self.prompt_factory.audit_url_content_prompt(
            url=url,
            topic=topic,
            content=content,
            ad_score=ad_score,
            agent_state=self.state,
        )

        try:
            res = await self.llm_client.single_response(
                prompt=prompt,
                output_metadata=True,
                temperature=1,
            )
            content_str = extract_content_from_response(res)
            data = extract_json_from_response(content_str)

            verdict = VerdictType(data.get("verdict", "REJECT"))

            audit = AuditDecision(
                url=url,
                verdict=verdict,
                verdict_confidence=data.get("verdict_confidence", 0.5),
                primary_reason=data.get("primary_reason", "No reason"),
                website_type=data.get("website_type", "Unknown"),
                specialization_score=data.get("specialization_score", 0.0),
                content_age_months=data.get("content_age_months"),
                reasoning_detail=data.get("decision_tree_path", ""),
            )
            audit.topic = topic

            if audit.verdict == VerdictType.APPROVE:
                self.state.urls_approved += 1
            else:
                if self.state.common_rejection_reason != audit.website_type:
                    self.state.common_rejection_reason = audit.website_type

            return audit
        except Exception as e:
            print(f"❌ Audit failed: {e}")
            return AuditDecision(
                url=url,
                verdict=VerdictType.REJECT,
                verdict_confidence=0.0,
                primary_reason=f"Audit error: {str(e)}",
                website_type="ERROR",
                specialization_score=0.0,
            )

    async def judge_review(
        self,
        audit: AuditDecision,
        content: str,
    ) -> JudgeReview | None:
        """Judge recenzuje auditní rozhodnutí (optional pro hraniční případy)."""
        if not (0.3 <= audit.verdict_confidence <= 0.7):
            return None

        prompt = self.prompt_factory.judge_review_prompt(audit, content)

        try:
            res = await self.judge_client.single_response(
                prompt=prompt,
                output_metadata=True,
                temperature=1,
            )
            content_str = extract_content_from_response(res)
            data = extract_json_from_response(content_str)

            review = JudgeReview(
                audit_verdict=audit.verdict,
                judge_verdict=VerdictType(data.get("judge_verdict", "UNCLEAR")),
                judge_confidence=data.get("judge_confidence", 0.5),
                agrees_with_auditor=data.get("agrees_with_auditor", True),
                judge_reasoning=data.get("judge_reasoning", ""),
                recommended_action=data.get(
                    "recommended_action",
                    "NEEDS_MANUAL_REVIEW",
                ),
            )

            self.metrics.record_judge_review(review)
            return review
        except Exception as e:
            print(f"❌ Judge review failed: {e}")
            return None

    async def process_topic(
        self,
        topic: str,
        urls_per_topic: int = 30,
    ) -> List[AuditDecision]:
        """Zpracuj jedno téma (celý pipeline pro jedno topic)."""

        self.state.current_topic = topic
        self.state.urls_processed = 0
        self.state.urls_approved = 0
        self.state.common_rejection_reason = ""

        print(f"\n{'=' * 60}")
        print(f"📌 TOPIC: {topic.upper()}")
        print(f"{'=' * 60}")

        # 1) Získání URL – SerpApi pokud je k dispozici, jinak LLM seed-URLs
        if self.search_provider is not None:
            print("  🔎 [Search] Fetching URLs via SerpApi...")
            seed_urls = await self.search_provider.search(topic, urls_per_topic)
        else:
            print("  🔎 [Search] Generating URLs via LLM prompt...")
            seed_urls = await self.generate_seed_urls(topic, urls_per_topic)

        unique_urls = list(set(seed_urls))
        topic_results: List[AuditDecision] = []

        # 2) Procesuj jednotlivé URL
        for idx, url in enumerate(unique_urls, 1):
            self.state.current_batch_index = idx
            self.state.urls_processed = idx

            if any(
                pattern in url
                for pattern in ["facebook.com", "instagram.com", "twitter.com"]
            ):
                continue

            domain = urlparse(url).netloc.replace("www.", "")
            if domain in self.state.processed_domains:
                continue

            print(f" 🔎 [{idx}/{len(unique_urls)}] {url}...", end=" ", flush=True)

            success, final_url, content, ad_score = get_page_content_smart(url)
            if not success:
                print("❌ Nedostupné")
                continue

            # canonicalizace URL na blog/homepage
            canonical_url = canonicalize_blog_url(final_url)

            self.state.processed_domains.add(domain)

            if blacklist.is_blacklisted(canonical_url):
                print("🚫 BLACKLISTED")
                continue
                
            audit = await self.audit_url_content(canonical_url, topic, content, ad_score)

            judge = await self.judge_review(audit, content)

            if judge and judge.recommended_action == "APPROVE_FINAL":
                audit.verdict = VerdictType.APPROVE
            elif judge and judge.recommended_action == "REJECT_FINAL":
                audit.verdict = VerdictType.REJECT

            icon = "✅" if audit.verdict == VerdictType.APPROVE else "⛔"
            print(f"{icon} {audit.website_type}")

            self.metrics.record_url_attempt(topic, canonical_url, audit)
            topic_results.append(audit)
            self.results.append(audit)

        return topic_results

    async def run_full_pipeline(
        self,
        num_topics: int = 10,
        urls_per_topic: int = 30,
        memory: TopicsMemory = None,
    ) -> pd.DataFrame:
        """Spusť celý pipeline: topics → URLs → audit → DF."""
        print("\n🚀 [Agent] Starting full pipeline...\n")

        # 1) Generování témat
        topics = await self.generate_niche_topics(num_topics)
        if not topics:
            print("❌ No topics generated")
            return pd.DataFrame()

        # Filtruj témata která už jsou v paměti
        if memory:
            existing = memory.get_topics()
            topics = [t for t in topics if t not in existing]
            print(f"✅ Generated {len(topics)} new topics (skipped {len(existing)} from memory)")
        else:
            print(f"✅ Generated {len(topics)} topics")

        if not topics:
            print("⚠️  All topics already in memory")
            return pd.DataFrame()

        print(f"  Topics: {', '.join(topics[:5])}...\n")

        # 2) Zpracování témat
        all_results: List[AuditDecision] = []
        for topic in topics:
            results = await self.process_topic(topic, urls_per_topic)
            all_results.extend(results)

            # Ulož do paměti hned po zpracování tématu
            if memory and results:
                urls = [r.url for r in results if r.verdict == VerdictType.APPROVE]
                if urls:
                    memory.add_topic(topic, urls, {"count": len(results)})

        # 3) Postprocessing → DataFrame
        df = self._postprocess_results(all_results)

        # 4) Výpis statistik
        self._print_final_stats(df)

        return df

    def _postprocess_results(self, results: List[AuditDecision]) -> pd.DataFrame:
        """Zpracuj výsledky do DataFrame."""
        if not results:
            return pd.DataFrame()

        rows = []
        for r in results:
            if r.verdict != VerdictType.APPROVE:
                continue
            domain = urlparse(r.url).netloc.replace("www.", "")
            rows.append(
                {
                    "topic": r.url,
                    "url": r.url,
                    "domain": domain,
                    "website_type": r.website_type,
                    "specialization_score": r.specialization_score,
                    "verdict_confidence": r.verdict_confidence,
                }
            )

        return pd.DataFrame(rows)

    def _print_final_stats(self, df: pd.DataFrame) -> None:
        """Jednoduchý výpis statistik nad výsledky."""
        if df.empty:
            print("❌ No approved URLs")
            return

        print("\n📊 FINAL STATS")
        print(f"  Total approved URLs: {len(df)}")
        print(f"  Avg specialization score: {df['specialization_score'].mean():.2f}")
        print(f"  By website_type:\n{df['website_type'].value_counts()}")

# 6. USAGE EXAMPLE

In [77]:
# Krok 1: Vytvoř memory
memory = TopicsMemory("topics_memory.json")
memory.show_stats()


📚 Memory Stats:
  Total topics: 0
  Total URLs: 0


In [82]:
# CELL: Seed topics - editoři
SEED_TOPICS = [
    "Neziskové organizace",
    "Poezie",
    "Jeskyňáři",
    "Památky a péče o ně",
    "Nemocnice",
    "Nekomerční kultura (Host, Tvar, Nedělní chvilka poezie)",
    "LGBTI+",
    "Univerzity a vysoké školství",
    "Jezdectví",
    "Církev",
    "Fotografie",
    "Literatura",
    "Vzdělávání pro děti",
    "Běh",
    "Biatlon",
    "Sjezdové lyžování",
    "Šití/Pletení",
    "Florbal",
    "Jóga",
    "HipHop",
    "Mediální gramotnost",
    "Horory",
    "Metal",
    "Rybaření",
    "Myslivost",
    "Turistika",
    "Únikovky",
    "Meditace/mindfullness",
    "Kempování",
    "Grilování",
    "Modelářství",
    "Audioknihy",
    "Mince",
    "Známky",
    "Akvaristika",
    "Teraristika",
    "Zpěv",
    "Inline bruslení",
]

# Přidej je do paměti
memory = TopicsMemory("topics_memory.json")

for topic in SEED_TOPICS:
    if not memory.has_topic(topic):
        memory.add_topic(topic, [], {"source": "editor_seed"})

memory.show_stats()
print(f"\n✅ {len(SEED_TOPICS)} seed topics added to memory")

  💾 [Neziskové organizace] Saved to memory (0 URLs)
  💾 [Poezie] Saved to memory (0 URLs)
  💾 [Jeskyňáři] Saved to memory (0 URLs)
  💾 [Památky a péče o ně] Saved to memory (0 URLs)
  💾 [Nemocnice] Saved to memory (0 URLs)
  💾 [Nekomerční kultura (Host, Tvar, Nedělní chvilka poezie)] Saved to memory (0 URLs)
  💾 [LGBTI+] Saved to memory (0 URLs)
  💾 [Univerzity a vysoké školství] Saved to memory (0 URLs)
  💾 [Jezdectví] Saved to memory (0 URLs)
  💾 [Církev] Saved to memory (0 URLs)
  💾 [Fotografie] Saved to memory (0 URLs)
  💾 [Literatura] Saved to memory (0 URLs)
  💾 [Vzdělávání pro děti] Saved to memory (0 URLs)
  💾 [Běh] Saved to memory (0 URLs)
  💾 [Biatlon] Saved to memory (0 URLs)
  💾 [Sjezdové lyžování] Saved to memory (0 URLs)
  💾 [Šití/Pletení] Saved to memory (0 URLs)
  💾 [Florbal] Saved to memory (0 URLs)
  💾 [Jóga] Saved to memory (0 URLs)
  💾 [HipHop] Saved to memory (0 URLs)
  💾 [Mediální gramotnost] Saved to memory (0 URLs)
  💾 [Horory] Saved to memory (0 URLs)
  💾 [Meta

In [87]:
import json
from pathlib import Path
from urllib.parse import urlparse

class Blacklist:
    """Jednoduchý blacklist na dezinformace, konspiraci, junk news."""
    
    def __init__(self, filepath: str = "blacklist.json"):
        self.filepath = Path(filepath)
        self.blacklisted_urls = set()
        self._load()
    
    def _load(self):
        """Načti blacklist z JSON."""
        if not self.filepath.exists():
            print(f"⚠️  {self.filepath} not found")
            return
        
        try:
            with open(self.filepath, "r", encoding="utf-8") as f:
                data = json.load(f)
            
            # Struktura: {"konspirace_a_dezinformace": [...], "junk_news": [...]}
            if isinstance(data, dict):
                for category, urls in data.items():
                    if isinstance(urls, list):
                        self.blacklisted_urls.update(urls)
                        print(f"  📋 {category}: {len(urls)} URLs")
            
            print(f"\n✅ Blacklist loaded: {len(self.blacklisted_urls)} total URLs")
        
        except Exception as e:
            print(f"❌ Blacklist load failed: {e}")
    
    def is_blacklisted(self, url: str) -> bool:
        """Zkontroluj, zda je URL na blacklistu."""
        # Normalizuj URL (bez trailing slash pro srovnání)
        url_normalized = url.rstrip("/").lower()
        
        # Srovnávej proti všem blacklistovaným URL
        for blacklisted in self.blacklisted_urls:
            blacklisted_normalized = blacklisted.rstrip("/").lower()
            
            # Přesná shoda nebo shoda domény
            if url_normalized == blacklisted_normalized:
                return True
            
            # Také zkontroluj jen doménu (ac24.cz se shoduje s https://ac24.cz)
            url_domain = urlparse(url).netloc.lower()
            blacklisted_domain = urlparse(blacklisted).netloc.lower()
            
            if url_domain and blacklisted_domain and url_domain == blacklisted_domain:
                return True
        
        return False

# Inicializace
blacklist = Blacklist("blacklist.json")

  📋 konspirace_a_dezinformace: 22 URLs
  📋 junk_news: 1176 URLs

✅ Blacklist loaded: 1196 total URLs


In [83]:
serp_provider = SerpApiSearchProvider(api_key=SERPAPI_KEY)
agent = WebFinderAgent(
    api_key=LLM_API_KEY,
    model="gpt-5",
    judge_model="gpt-5",
    search_provider=serp_provider,
)

In [102]:
# Krok 3: Spusť pipeline s pamětí a načíst BL
blacklist = Blacklist("blacklist.json")

df = await agent.run_full_pipeline(
    num_topics=10,
    urls_per_topic=10,
    memory=memory,  
)

  📋 konspirace_a_dezinformace: 22 URLs
  📋 junk_news: 1176 URLs

✅ Blacklist loaded: 1196 total URLs

🚀 [Agent] Starting full pipeline...

🧠 [Agent] Generating 10 niche topics...
✅ Generated 10 new topics (skipped 80 from memory)
  Topics: Simracing, Astrofotografie, Domácí vaření piva (homebrewing), Stavba mechanických klávesnic, FPV závody dronů...


📌 TOPIC: SIMRACING
  🔎 [Search] Fetching URLs via SerpApi...
  🔎 [SerpApi] Searching: 'Simracing'
     API Key: 55ad0f604d92b00...
     Page 1: Requesting Simracing...
       ✅ Got 8 results
     📊 Total found: 8 URLs

 🔎 [2/8] https://www.advancedsimracing.com/?srsltid=AfmBOoqLVcv9ng8-14vq32MZLdnvrXo2ZeNKgxL0NiZig8w-UQj_XsX5... ⛔ Eshop
 🔎 [3/8] https://simagic.com/... ⛔ Eshop
 🔎 [4/8] https://www.simracing.gp/... ✅ Corporate
 🔎 [6/8] https://www.apexsimracing.com/?srsltid=AfmBOop9JSnJm8ndfb16HuPVx0xw1jCiGwrG7aijKw5bolKeFY_s52pg... ⛔ Eshop
 🔎 [7/8] https://nextlevelracing.com/racing-cockpits/?srsltid=AfmBOoqICigheyRb8y8X51dLEzPjo-tRtqrL3